## 🔹 Task 1: Training a Threshold Logic Unit (TLU)

Given a Threshold Logic Unit with:

- Initial weights: w = 0
- Threshold: θ = 1.0
- Learning rate: η = 0.3

Objective:
Train the TLU to represent the Boolean identity function:

- y = x

Instructions:
Complete the following table to describe the training process:

- Iteration step
- Input (x)
- Output before training (o)
- Weighted input (xw)
- Expected output (y)
- Error (o − y)
- Change in threshold (Δθ)
- Change in weights (Δw)
- Updated θ and w

📌 Note:

- You may use your implementation (Task 3)
- But calculate at least one step manually

| Step | x   | o   | xw  | y   | o−y | Δθ   | Δw   | θ   | w   |
| ---- | --- | --- | --- | --- | --- | ---- | ---- | --- | --- |
| 1    | 0   | 0   | 0   | 0   | 0   | 0    | 0    | 1.0 | 0   |
| 2    | 1   | 0   | 0   | 1   | -1  | -0.3 | +0.3 | 0.7 | 0.3 |
| 3    | 1   | 0   | 0.3 | 1   | -1  | -0.3 | +0.3 | 0.4 | 0.6 |
| 4    | 1   | 1   | 0.6 | 1   | 0   | 0    | 0    | 0.4 | 0.6 |


# 🔹 Task 2: XOR Classification

- Test graphically whether a linear function can perfectly classify the Boolean XOR function

The XOR function is not linearly separable, therefore a single Threshold Logic Unit cannot represent it.


# 🔹 Task 3: TLU Implementation

Objective:
Implement a Threshold Logic Unit (TLU) in Python 3

The TLU should approximate the following Boolean functions:

- NOT
- Identity
- AND
- OR
- Implication
- Bi-Implication

Given:

- Learning rate: η = 0.3
- Initial weights: wi = 0 (for all weights)
- Threshold: θ = 1.0

Instructions:
Complete the table for each Boolean function:

- Boolean function
- Number of training steps
- Final error after termination


In [3]:
import numpy as np

class TLU:
    def __init__(self, n_inputs):
        self.weights = np.zeros(n_inputs)
        self.theta = 1.0
        self.lr = 0.3

    def predict(self, x):
        s = np.dot(self.weights, x)
        if s >= self.theta:
            return 1
        else:
            return 0

    def train(self, X, y):
        steps = 0
        max_epochs = 1000

        for _ in range(max_epochs):
            error_count = 0

            for i in range(len(X)):
                x = X[i]
                target = y[i]

                output = self.predict(x)
                error = output - target

                if error != 0:
                    error_count += 1

                    self.weights = self.weights - self.lr * error * x
                    self.theta = self.theta + self.lr * error

                steps += 1

            if error_count == 0:
                break

        return steps, error_count




functions = {
    "Identity": (
        np.array([[0],[1]]),
        np.array([0,1])
    ),
    "NOT": (
        np.array([[0],[1]]),
        np.array([1,0])
    ),
    "AND": (
        np.array([[0,0],[0,1],[1,0],[1,1]]),
        np.array([0,0,0,1])
    ),
    "OR": (
        np.array([[0,0],[0,1],[1,0],[1,1]]),
        np.array([0,1,1,1])
    ),
    "Implication": (
        np.array([[0,0],[0,1],[1,0],[1,1]]),
        np.array([1,1,0,1])
    ),
    "Bi-Implication": (
        np.array([[0,0],[0,1],[1,0],[1,1]]),
        np.array([1,0,0,1])
    )
}




for name in functions:
    X, y = functions[name]

    model = TLU(len(X[0]))
    steps, error = model.train(X, y)

    if error == 0:
        status = "Yes"
    else:
        status = "No"

    print(name, "-> steps:", steps, " error:", error, " converged:", status)

Identity -> steps: 6  error: 0  converged: Yes
NOT -> steps: 12  error: 0  converged: Yes
AND -> steps: 16  error: 0  converged: Yes
OR -> steps: 12  error: 0  converged: Yes
Implication -> steps: 20  error: 0  converged: Yes
Bi-Implication -> steps: 4000  error: 4  converged: No


Additional Question:

- How does the training output change when:
  - changing weight initialization
  - changing learning rate

Effect of weight initialization:
Weight initialization determines the starting point of the learning process in a TLU. If the initial weights are changed, the model begins learning from a different position, which can affect the number of iterations required to reach the correct solution. Some initial values may lead to faster convergence, while others may slow down the learning process. However, for linearly separable problems such as AND, OR, and Identity, the final result will still be correct regardless of the initial weights. For non-linearly separable problems like XOR or XNOR, changing the initial weights does not help, and the model will still fail to converge.

Effect of learning rate:
The learning rate controls how much the weights and threshold are updated during each step of training. A small learning rate results in slow but stable learning, requiring more iterations to converge. A moderate learning rate provides a good balance between speed and stability and is generally preferred. In contrast, a very large learning rate can cause the model to overshoot the correct solution, leading to unstable behavior and possible failure to converge. If the learning rate is zero, no updates occur at all, and the model cannot learn.


# 🔹 Task 4: Applying TLU to Titanic Dataset

- Apply your TLU model to the survival prediction task
- Use the same features as in the previous exercise

Compare the performance with:

- Decision Tree
- Guessing Model

Instructions:

- Use the same evaluation metrics as before
- Analyze how the TLU performs compared to other models

📌 Note

- Focus on understanding how TLU learns
- Compare linear vs non-linear problems (important for XOR)
- Observe performance differences between models


In [6]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier

X_train = np.load("../data/X_train.npy")
X_test  = np.load("../data/X_test.npy")
y_train = np.load("../data/y_train.npy")
y_test  = np.load("../data/y_test.npy")

print(X_train.shape, X_test.shape)

(712, 6) (179, 6)


In [ ]:

#Guessing Model

class GuessingModel:
    def fit(self, X, y):
        self.p = np.mean(y)

    def predict(self, X):
        return np.random.choice([0,1], size=len(X), p=[1-self.p, self.p])

In [8]:
tlu = TLU(X_train.shape[1])
steps, err = tlu.train(X_train, y_train)

dt = DecisionTreeClassifier()
dt.fit(X_train, y_train)

gm = GuessingModel()
gm.fit(X_train, y_train)

print("training done")

training done


In [9]:
def accuracy(y_true, y_pred):
    return np.sum(y_true == y_pred) / len(y_true)

def f1_score(y_true, y_pred):
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))

    if tp + fp == 0:
        precision = 0
    else:
        precision = tp / (tp + fp)

    if tp + fn == 0:
        recall = 0
    else:
        recall = tp / (tp + fn)

    if precision + recall == 0:
        return 0

    return 2 * precision * recall / (precision + recall)

In [12]:
# Prediction 
preds_tlu = np.array([tlu.predict(x) for x in X_test])
preds_dt  = dt.predict(X_test)
preds_gm  = gm.predict(X_test)

print("TLU:", accuracy(y_test, preds_tlu), f1_score(y_test, preds_tlu))
print("Decision Tree:", accuracy(y_test, preds_dt), f1_score(y_test, preds_dt))
print("Guessing:", accuracy(y_test, preds_gm), f1_score(y_test, preds_gm))

print("TLU steps:", steps, "error:", err)

TLU: 0.770949720670391 0.6870229007633587
Decision Tree: 0.7988826815642458 0.7142857142857143
Guessing: 0.4748603351955307 0.3648648648648649
TLU steps: 712000 error: 203


The TLU performs better than the guessing model because it learns from input features. However, it performs worse than the decision tree because it can only model linear relationships, while the Titanic dataset contains more complex patterns. The decision tree performs best because it can capture non-linear relationships between features.
